# Deep Learning Models

## MLP Model

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.interpolate import interp1d
from scipy.signal import find_peaks # <-- Make sure this is imported

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import experiment_tracker 

# --- Define common parameters ---
all_battery_data_EIS = pd.read_csv('all_battery_data_with_EIS_params.csv')
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']

# ===================================================================
# PHASE 0: ADVANCED FEATURE ENGINEERING (from your first script)
# ===================================================================

print("--- Phase 0: Preparing Data & Engineering Features ---")

# 1. Clean and align all data
# We use 'SoC' as it is used in the feat_SoC line below
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH', 'SoC'] 
).reset_index(drop=True)

# 2. Define fixed frequencies for features
fixed_freqs = np.logspace(-2, 4, 50) # 50 points from 0.01 Hz to 10 kHz

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    
    # --- Get the SoC feature ---
    feat_SoC = row['SoC'] # This is your contextual feature
    
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)') 
        
        f_x = raw_df['Frequency(Hz)'].values
        R = raw_df['R(ohm)'].values
        X = raw_df['X(ohm)'].values
        Z_imag_neg = -X
        
        # --- A. Calculate 6 Peak-Finding Features ---
        feat_R_ohmic = R[-1] 
        feat_R_low = R[0]
        indices, _ = find_peaks(Z_imag_neg, prominence=1e-4) 
        
        if len(indices) == 2:
            feat_Peak_LF_Height = Z_imag_neg[indices[0]]
            feat_Peak_LF_Freq = f_x[indices[0]]
            feat_Peak_HF_Height = Z_imag_neg[indices[1]]
            feat_Peak_HF_Freq = f_x[indices[1]]
        elif len(indices) == 1:
            feat_Peak_LF_Height = Z_imag_neg[indices[0]]
            feat_Peak_LF_Freq = f_x[indices[0]]
            feat_Peak_HF_Height = 0.0 
            feat_Peak_HF_Freq = 0.0
        else:
            feat_Peak_LF_Height = 0.0
            feat_Peak_LF_Freq = 0.0
            feat_Peak_HF_Height = 0.0
            feat_Peak_HF_Freq = 0.0

        plot_features_array = np.array([
            feat_R_ohmic, feat_R_low, 
            feat_Peak_LF_Height, feat_Peak_LF_Freq,
            feat_Peak_HF_Height, feat_Peak_HF_Freq
        ])
        
        # --- B. Calculate 2 Phase-Based Features ---
        Z_phase_deg = np.angle(R + 1j * X, deg=True)
        feat_phase_min = np.min(Z_phase_deg)
        feat_phase_min_freq = f_x[np.argmin(Z_phase_deg)]
        
        phase_features_array = np.array([feat_phase_min, feat_phase_min_freq])

        # --- C. Calculate 100 Interpolated Features ---
        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        R_features = interp_R(fixed_freqs)
        X_features = interp_X(fixed_freqs)
        
        # --- D. Combine ALL features (100 + 6 + 2 + 1 = 109 features) ---
        features = np.concatenate([
            R_features, 
            X_features, 
            plot_features_array, 
            phase_features_array,
            np.array([feat_SoC]) # Add SoC
        ])
        eis_features_list.append(features)
        
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None)

# 4. Create the final aligned X and y DataFrames
r_cols = [f'R_{freq:.2f}Hz' for freq in fixed_freqs]
x_cols = [f'X_{freq:.2f}Hz' for freq in fixed_freqs]
plot_feature_names = [
    'R_ohmic', 'R_low', 'Peak_LF_Height', 'Peak_LF_Freq', 
    'Peak_HF_Height', 'Peak_HF_Freq'
]
phase_feature_names = ['Phase_Min', 'Phase_Min_Freq']
context_feature_names = ['SoC'] # Your new feature

all_feature_names = (
    r_cols + x_cols + 
    plot_feature_names + 
    phase_feature_names + 
    context_feature_names
)

X_features_df = pd.DataFrame(eis_features_list, columns=all_feature_names)
y_params_df = df_cleaned[param_columns]
y_soh_df = df_cleaned[['SoH']]

# 5. Drop any rows that failed feature engineering
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

print(f"Data ready: X_features ({X_features_final.shape}), y_params ({y_params_final.shape}), y_soh ({y_soh_final.shape})")

# ===================================================================
# PHASE 1: PYTORCH SETUP (Dataset & Model Architecture)
# ===================================================================

print("\n--- Phase 1: Setting up PyTorch components ---")

# 1. Scale features
scaler_X = StandardScaler()
scaler_y_params = StandardScaler()

X_scaled = scaler_X.fit_transform(X_features_final)
y_params_scaled = scaler_y_params.fit_transform(y_params_final)
y_soh_values = y_soh_final.values # No scaling needed for SoH (already 0-1)

# 2. Train/Test Split
(X_train, X_val, 
 y_params_train, y_params_val, 
 y_soh_train, y_soh_val) = train_test_split(
    X_scaled, y_params_scaled, y_soh_values, test_size=0.2, random_state=42
)

# 3. Custom PyTorch Dataset
class EISDataset(Dataset):
    def __init__(self, features, params, soh):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y_params = torch.tensor(params, dtype=torch.float32)
        self.y_soh = torch.tensor(soh, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_params[idx], self.y_soh[idx]

train_dataset = EISDataset(X_train, y_params_train, y_soh_train)
val_dataset = EISDataset(X_val, y_params_val, y_soh_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 4. Define the Multi-Head Model
class MultiHeadEISModel(nn.Module):
    def __init__(self, input_size, num_params):
        super(MultiHeadEISModel, self).__init__()
        
        # Shared Body
        self.shared_body = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Head 1: Parameter Prediction
        self.param_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_params) # No activation (linear output for regression)
        )
        
        # Head 2: SoH Prediction
        self.soh_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1), # No activation (linear output for regression)
            nn.Sigmoid() # Add Sigmoid to bound output between 0 and 1
        )

    def forward(self, x):
        # Pass input through the shared body
        shared_output = self.shared_body(x)
        
        # Pass shared output to each head
        params_pred = self.param_head(shared_output)
        soh_pred = self.soh_head(shared_output)
        
        return params_pred, soh_pred

# ===================================================================
# PHASE 2: MODEL TRAINING
# ===================================================================

print("\n--- Phase 2: Starting Model Training ---")

# 1. Setup Model, Loss, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

INPUT_SIZE = X_features_final.shape[1] # This will now be 109
NUM_PARAMS = y_params_final.shape[1]
NUM_EPOCHS = 100 # Increase this for better results
LEARNING_RATE = 0.001

print(f"Model Input Size: {INPUT_SIZE}") # Will print 109

model = MultiHeadEISModel(INPUT_SIZE, NUM_PARAMS).to(device)
loss_params_fn = nn.MSELoss()
loss_soh_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Define your loss weights
alpha = 1000.0  # Make the SoH loss "louder"
beta = 1.0 # Weight for Parameter loss
history = {
    'train_loss': [], 
    'val_total': [], 
    'val_soh': [], 
    'val_params': []
}
# 3. Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    
    for features, true_params, true_soh in train_loader:
        features = features.to(device)
        true_params = true_params.to(device)
        true_soh = true_soh.to(device)
        
        # --- Forward Pass ---
        pred_params, pred_soh = model(features)
        
        # --- Calculate Combined Loss ---
        loss_params = loss_params_fn(pred_params, true_params)
        loss_soh = loss_soh_fn(pred_soh, true_soh)
        
        loss_total = (alpha * loss_soh) + (beta * loss_params)
        
        # --- Backward Pass ---
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()
        
        total_train_loss += loss_total.item()

    # --- Validation ---
    model.eval()
    total_val_loss = 0
    total_val_soh_loss = 0
    total_val_params_loss = 0
    
    with torch.no_grad():
        for features, true_params, true_soh in val_loader:
            features = features.to(device)
            true_params = true_params.to(device)
            true_soh = true_soh.to(device)
            
            pred_params, pred_soh = model(features)
            
            loss_params = loss_params_fn(pred_params, true_params)
            loss_soh = loss_soh_fn(pred_soh, true_soh)
            loss_total = (alpha * loss_soh) + (beta * loss_params)
            
            total_val_loss += loss_total.item()
            total_val_soh_loss += loss_soh.item()
            total_val_params_loss += loss_params.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    avg_soh_loss = total_val_soh_loss / len(val_loader)
    avg_params_loss = total_val_params_loss / len(val_loader)
     # --- ADDED: Save losses to history ---
    history['train_loss'].append(avg_train_loss)
    history['val_total'].append(avg_val_loss)
    history['val_soh'].append(avg_soh_loss)
    history['val_params'].append(avg_params_loss)
    # ---
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} (SoH: {avg_soh_loss:.6f}, Params: {avg_params_loss:.6f})")

print("\n--- Training Complete ---")

# ===================================================================
# PHASE 3: FINAL EVALUATION
# ===================================================================
model.eval()
with torch.no_grad():
    # Predict on the entire validation set
    all_features = torch.tensor(X_val, dtype=torch.float32).to(device)
    true_soh_torch = torch.tensor(y_soh_val, dtype=torch.float32).to(device)
    
    pred_params_scaled, pred_soh = model(all_features)
    
    # --- De-scale the predictions to be human-readable ---
    pred_params = scaler_y_params.inverse_transform(pred_params_scaled.cpu().numpy())
    
    # --- De-scale the true parameters for comparison (FIX) ---
    true_params_descaled = scaler_y_params.inverse_transform(y_params_val)
    
    # --- Calculate Final Errors ---
    soh_rmse = np.sqrt(mean_squared_error(y_soh_val, pred_soh.cpu().numpy()))
    
    print(f"\n--- Final Model Evaluation ---")
    print(f"SoH Prediction RMSE: {soh_rmse*100:.2f}% SoH")
    final_rmse_dict = {'SoH': soh_rmse}
    # Calculate RMSE for each parameter
    for i, name in enumerate(param_columns):
        # Compare de-scaled true values vs. de-scaled predicted values
        param_rmse = np.sqrt(mean_squared_error(true_params_descaled[:, i], pred_params[:, i]))
        print(f"  - {name} RMSE: {param_rmse:.4f}")
        final_rmse_dict[name] = param_rmse
exp_name = "Simple_MLP_109features_v1" 
model_arch_name = "Simple_MLP"

try:
    print("\n--- Saving learning curves to JSON file ---")
    experiment_tracker.save_experiment_results(
        experiment_name=exp_name,
        model_architecture=model_arch_name,
        final_rmse=final_rmse_dict,
        learning_curves=history
    )
    print("--- JSON save complete. ---")
except Exception as e:
    print(f"--- !! WARNING: JSON tracker save failed: {e} !! ---")


--- Phase 0: Preparing Data & Engineering Features ---


Processing Spectra: 100%|██████████| 549/549 [00:00<00:00, 1974.79it/s]


Data ready: X_features ((549, 109)), y_params ((549, 6)), y_soh ((549, 1))

--- Phase 1: Setting up PyTorch components ---

--- Phase 2: Starting Model Training ---
Using device: cpu
Model Input Size: 109
Epoch [01/100] - Train Loss: 80.890318 | Val Loss: 15.275478 (SoH: 0.014192, Params: 1.083038)
Epoch [02/100] - Train Loss: 14.049129 | Val Loss: 13.885987 (SoH: 0.012917, Params: 0.969388)
Epoch [03/100] - Train Loss: 8.427978 | Val Loss: 6.591458 (SoH: 0.005614, Params: 0.977675)
Epoch [04/100] - Train Loss: 4.852537 | Val Loss: 4.324857 (SoH: 0.003314, Params: 1.010856)
Epoch [05/100] - Train Loss: 3.745430 | Val Loss: 3.464091 (SoH: 0.002479, Params: 0.985096)
Epoch [06/100] - Train Loss: 3.426442 | Val Loss: 3.148253 (SoH: 0.002180, Params: 0.968217)
Epoch [07/100] - Train Loss: 3.460994 | Val Loss: 3.485280 (SoH: 0.002533, Params: 0.952249)
Epoch [08/100] - Train Loss: 3.049286 | Val Loss: 3.342332 (SoH: 0.002413, Params: 0.929610)
Epoch [09/100] - Train Loss: 2.798183 | Val Los

## CNN-RNN Model

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# --- ADDED: Import the tracker ---
import experiment_tracker 

# ===================================================================
# --- COMMON PARAMETERS ---
# ===================================================================
all_battery_data_EIS = pd.read_csv('all_battery_data_with_EIS_params.csv')
# --- REWRITTEN: Predicting all 6 parameters ---
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1'] # 6 parameters
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']

# ===================================================================
# PHASE 0: DATA & FEATURE ENGINEERING
# ===================================================================

print("--- Phase 0: Preparing Data & Engineering Features ---")

# 1. Clean and align all data
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH', 'SoC']
).reset_index(drop=True)

# 2. Define fixed frequencies for features
num_freq_points = 50
fixed_freqs = np.logspace(-2, 4, num_freq_points) # 50 points

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)') 
        f_x, R, X = raw_df['Frequency(Hz)'].values, raw_df['R(ohm)'].values, raw_df['X(ohm)'].values
        Z_imag_neg = -X
        
        feat_R_ohmic = R[-1] 
        feat_R_low = R[0]
        indices, _ = find_peaks(Z_imag_neg, prominence=1e-4) 
        if len(indices) == 2:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = Z_imag_neg[indices[1]], f_x[indices[1]]
        elif len(indices) == 1:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0
        else:
            feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0, 0.0, 0.0
        
        # 6 scalar plot features
        plot_features_array = np.array([feat_R_ohmic, feat_R_low, feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq])
        
        Z_phase_deg = np.angle(R + 1j * X, deg=True)
        feat_phase_min, feat_phase_min_freq = np.min(Z_phase_deg), f_x[np.argmin(Z_phase_deg)]
        
        # 2 scalar phase features
        phase_features_array = np.array([feat_phase_min, feat_phase_min_freq])

        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        
        # 50 + 50 = 100 spectrum features
        R_features, X_features = interp_R(fixed_freqs), interp_X(fixed_freqs)
        
        # --- Correct 8 scalar features (108 total) ---
        features = np.concatenate([R_features, X_features, plot_features_array, phase_features_array])
        eis_features_list.append(features)
        
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None)

# 4. Create final aligned DataFrames
r_cols = [f'R_{i}' for i in range(num_freq_points)]
x_cols = [f'X_{i}' for i in range(num_freq_points)]
plot_feature_names = ['R_ohmic', 'R_low', 'Peak_LF_Height', 'Peak_LF_Freq', 'Peak_HF_Height', 'Peak_HF_Freq']
phase_feature_names = ['Phase_Min', 'Phase_Min_Freq']

all_feature_names = r_cols + x_cols + plot_feature_names + phase_feature_names # 108 total

X_features_df = pd.DataFrame(eis_features_list, columns=all_feature_names)
y_params_df = df_cleaned[param_columns] # Now (N, 6)
y_soh_df = df_cleaned[['SoH']]

# 5. Drop failed rows
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

# --- 6. Split X into spectrum (100) and scalar (8) features ---
spec_cols = r_cols + x_cols
X_spec_final = X_features_final[spec_cols]
scalar_cols = plot_feature_names + phase_feature_names # 8 features
X_scalar_final = X_features_final[scalar_cols]

print(f"Data ready: X_spectrum ({X_spec_final.shape}), X_scalar ({X_scalar_final.shape})")
print(f"Targets ready: y_params ({y_params_final.shape}), y_soh ({y_soh_final.shape})")

# ===================================================================
# PHASE 1: PYTORCH SETUP (Full Multi-Head Model)
# ===================================================================

print("\n--- Phase 1: Setting up PyTorch components ---")

# 1. Scale each DataFrame separately
scaler_X_spec = StandardScaler()
scaler_X_scalar = StandardScaler()
scaler_y_params = StandardScaler() # This will be fit on 6 params

X_spec_scaled = scaler_X_spec.fit_transform(X_spec_final)
X_scalar_scaled = scaler_X_scalar.fit_transform(X_scalar_final)
y_params_scaled = scaler_y_params.fit_transform(y_params_final) # (N, 6)
y_soh_values = y_soh_final.values 

# 2. Train/Test Split
(X_spec_train, X_spec_val,
 X_scalar_train, X_scalar_val,
 y_params_train, y_params_val, 
 y_soh_train, y_soh_val) = train_test_split(
    X_spec_scaled, X_scalar_scaled, y_params_scaled, y_soh_values, 
    test_size=0.2, random_state=42
)
print(f"Train/Test split: X_scalar_train ({X_scalar_train.shape}), y_params_train ({y_params_train.shape})")

# 3. Custom PyTorch Dataset
class EISDataset(Dataset):
    def __init__(self, spec_features, scalar_features, params, soh):
        self.X_spec = torch.tensor(spec_features, dtype=torch.float32)   # (N, 100)
        self.X_scalar = torch.tensor(scalar_features, dtype=torch.float32) # (N, 8)
        self.y_params = torch.tensor(params, dtype=torch.float32)       # (N, 6)
        self.y_soh = torch.tensor(soh, dtype=torch.float32)             # (N, 1)

    def __len__(self):
        return len(self.X_spec)

    def __getitem__(self, idx):
        return (self.X_spec[idx], self.X_scalar[idx]), self.y_params[idx], self.y_soh[idx]

train_dataset = EISDataset(X_spec_train, X_scalar_train, y_params_train, y_soh_train)
val_dataset = EISDataset(X_spec_val, X_scalar_val, y_params_val, y_soh_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 4. --- REWRITTEN: Full Multi-Head (7-Tower) Model Architecture ---
class MultiHeadEISModel(nn.Module):
    def __init__(self, num_scalar_features, num_freq_points):
        super(MultiHeadEISModel, self).__init__()
        
        self.num_freq_points = num_freq_points # 50
        
        # --- Branch 1: CNN for Spectrum (Unchanged) ---
        self.cnn_branch = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2), # 50 -> 25
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2), # 25 -> 12
            nn.Flatten() 
        )
        
        # --- Branch 2: MLP for Scalar Features (Unchanged) ---
        self.scalar_branch = nn.Sequential(
            nn.Linear(num_scalar_features, 32), # Input is 8
            nn.ReLU(),
            nn.Linear(32, 64)
        )

        cnn_output_size = 32 * (num_freq_points // 4) # 32 * 12 = 384
        merged_feature_size = cnn_output_size + 64 # 384 + 64 = 448
        
        # --- NEW: 7 SEPARATE TOWERS ---
        # Each tower is a small, dedicated MLP to prevent gradient conflict
        
        # Tower for SoH
        self.soh_tower = nn.Sequential(
            nn.Linear(merged_feature_size, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
        
        # Tower for L1
        self.l1_tower = nn.Sequential(
            nn.Linear(merged_feature_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # Tower for R0
        self.r0_tower = nn.Sequential(
            nn.Linear(merged_feature_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # Tower for R1
        self.r1_tower = nn.Sequential(
            nn.Linear(merged_feature_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

        # --- NEW: Tower for CPE1_0 ---
        self.cpe1_0_tower = nn.Sequential(
            nn.Linear(merged_feature_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # Tower for CPE1_1
        self.cpe1_1_tower = nn.Sequential(
            nn.Linear(merged_feature_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # Tower for W1
        self.w1_tower = nn.Sequential(
            nn.Linear(merged_feature_size, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_spec, x_scalar):
        # 1. Get base features
        x_spec = x_spec.reshape(-1, 2, self.num_freq_points)
        out_cnn = self.cnn_branch(x_spec)
        out_scalar = self.scalar_branch(x_scalar)
        
        # 2. Merge features
        merged = torch.cat((out_cnn, out_scalar), dim=1) # Shape: (batch, 448)
        
        # 3. Feed merged features into EVERY tower
        pred_soh = self.soh_tower(merged)
        pred_l1 = self.l1_tower(merged)
        pred_r0 = self.r0_tower(merged)
        pred_r1 = self.r1_tower(merged)
        pred_cpe1_0 = self.cpe1_0_tower(merged) # Added
        pred_cpe1_1 = self.cpe1_1_tower(merged)
        pred_w1 = self.w1_tower(merged)
        
        # Return 7 separate tensors
        return pred_soh, pred_l1, pred_r0, pred_r1, pred_cpe1_0, pred_cpe1_1, pred_w1

# ===================================================================
# PHASE 2: MODEL TRAINING (Full Multi-Head Loop)
# ===================================================================

print("\n--- Phase 2: Starting Model Training ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Dynamically get input sizes
NUM_FREQ_POINTS = len(fixed_freqs) 
NUM_SCALAR_FEATURES = X_scalar_final.shape[1] # This will be 8
NUM_EPOCHS = 100 
LEARNING_RATE = 0.001 # Lowered Learning Rate

print(f"Number of Frequency Points: {NUM_FREQ_POINTS}")
print(f"Model Input Sizes: {NUM_FREQ_POINTS*2} (spectrum) + {NUM_SCALAR_FEATURES} (scalar)")

model = MultiHeadEISModel(NUM_SCALAR_FEATURES, NUM_FREQ_POINTS).to(device)
loss_fn = nn.MSELoss() # We can use one loss function for all
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- REWRITTEN: Individual Loss Weights for 7 tasks ---
weights = {
    'soh': 1000.0,  # High weight to make it visible
    'l1': 0.5,      # Easy task, lower weight
    'r0': 0.5,      # Easy task, lower weight
    'r1': 1.0,      # HARD task, high weight
    'cpe1_0': 1.0,  # HARD task, high weight
    'cpe1_1': 1.0,  # HARD task, high weight
    'w1': 0.5       # Easy task, lower weight
}
print(f"Using individual loss weights: {weights}")


# --- ADDED: Initialize history dictionary ---
history = {
    'train_loss': [], 'val_total': [], 'val_soh': [], 'val_l1': [], 'val_r0': [], 
    'val_r1': [], 'val_cpe1_0': [], 'val_cpe1_1': [], 'val_w1': []
}
# ---

# --- REWRITTEN: Training Loop for 7 tasks ---
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    
    for (features_spec, features_scalar), true_params, true_soh in train_loader:
        features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
        true_params, true_soh = true_params.to(device), true_soh.to(device)
        
        # Get true values for each head (y_params is (N, 6))
        true_l1 = true_params[:, 0].view(-1, 1)
        true_r0 = true_params[:, 1].view(-1, 1)
        true_r1 = true_params[:, 2].view(-1, 1)
        true_cpe1_0 = true_params[:, 3].view(-1, 1) # Added
        true_cpe1_1 = true_params[:, 4].view(-1, 1)
        true_w1 = true_params[:, 5].view(-1, 1)
        true_soh = true_soh.view(-1, 1) # Ensure shape is (batch, 1)
        
        # Get 7 separate predictions
        pred_soh, pred_l1, pred_r0, pred_r1, pred_cpe1_0, pred_cpe1_1, pred_w1 = model(features_spec, features_scalar)
        
        # Calculate 7 separate losses
        loss_soh = loss_fn(pred_soh, true_soh)
        loss_l1 = loss_fn(pred_l1, true_l1)
        loss_r0 = loss_fn(pred_r0, true_r0)
        loss_r1 = loss_fn(pred_r1, true_r1)
        loss_cpe1_0 = loss_fn(pred_cpe1_0, true_cpe1_0) # Added
        loss_cpe1_1 = loss_fn(pred_cpe1_1, true_cpe1_1)
        loss_w1 = loss_fn(pred_w1, true_w1)
        
        # Combine all losses with their weights
        loss_total = (weights['soh'] * loss_soh) + \
                     (weights['l1'] * loss_l1) + \
                     (weights['r0'] * loss_r0) + \
                     (weights['r1'] * loss_r1) + \
                     (weights['cpe1_0'] * loss_cpe1_0) + \
                     (weights['cpe1_1'] * loss_cpe1_1) + \
                     (weights['w1'] * loss_w1)
        
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()
        total_train_loss += loss_total.item()

    # --- Validation Loop ---
    model.eval()
    val_losses = {'total': 0, 'soh': 0, 'l1': 0, 'r0': 0, 'r1': 0, 'cpe1_0': 0, 'cpe1_1': 0, 'w1': 0}
    
    with torch.no_grad():
        for (features_spec, features_scalar), true_params, true_soh in val_loader:
            features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
            true_params, true_soh = true_params.to(device), true_soh.to(device)
            
            true_l1 = true_params[:, 0].view(-1, 1)
            true_r0 = true_params[:, 1].view(-1, 1)
            true_r1 = true_params[:, 2].view(-1, 1)
            true_cpe1_0 = true_params[:, 3].view(-1, 1) # Added
            true_cpe1_1 = true_params[:, 4].view(-1, 1)
            true_w1 = true_params[:, 5].view(-1, 1)
            true_soh = true_soh.view(-1, 1)
            
            pred_soh, pred_l1, pred_r0, pred_r1, pred_cpe1_0, pred_cpe1_1, pred_w1 = model(features_spec, features_scalar)
            
            loss_soh = loss_fn(pred_soh, true_soh)
            loss_l1 = loss_fn(pred_l1, true_l1)
            loss_r0 = loss_fn(pred_r0, true_r0)
            loss_r1 = loss_fn(pred_r1, true_r1)
            loss_cpe1_0 = loss_fn(pred_cpe1_0, true_cpe1_0) # Added
            loss_cpe1_1 = loss_fn(pred_cpe1_1, true_cpe1_1)
            loss_w1 = loss_fn(pred_w1, true_w1)
            
            loss_total = (weights['soh'] * loss_soh) + \
                         (weights['l1'] * loss_l1) + \
                         (weights['r0'] * loss_r0) + \
                         (weights['r1'] * loss_r1) + \
                         (weights['cpe1_0'] * loss_cpe1_0) + \
                         (weights['cpe1_1'] * loss_cpe1_1) + \
                         (weights['w1'] * loss_w1)
            
            val_losses['total'] += loss_total.item()
            val_losses['soh'] += loss_soh.item()
            val_losses['l1'] += loss_l1.item()
            val_losses['r0'] += loss_r0.item()
            val_losses['r1'] += loss_r1.item()
            val_losses['cpe1_0'] += loss_cpe1_0.item() # Added
            val_losses['cpe1_1'] += loss_cpe1_1.item()
            val_losses['w1'] += loss_w1.item()

    # Calculate averages
    avg_train_loss = total_train_loss / len(train_loader)
    for key in val_losses:
        val_losses[key] /= len(val_loader)
    
    # --- ADDED: Save losses to history ---
    history['train_loss'].append(avg_train_loss)
    history['val_total'].append(val_losses['total'])
    history['val_soh'].append(val_losses['soh'])
    history['val_l1'].append(val_losses['l1'])
    history['val_r0'].append(val_losses['r0'])
    history['val_r1'].append(val_losses['r1'])
    history['val_cpe1_0'].append(val_losses['cpe1_0'])
    history['val_cpe1_1'].append(val_losses['cpe1_1'])
    history['val_w1'].append(val_losses['w1'])
    # ---
    
    # New print statement to monitor all tasks
    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {val_losses['total']:.4f}")
        print(f"    Val SoH: {val_losses['soh']:.6f} | R1: {val_losses['r1']:.6f} | CPE1_0: {val_losses['cpe1_0']:.6f} | CPE1_1: {val_losses['cpe1_1']:.6f}")
        print(f"    L1: {val_losses['l1']:.6f} | R0: {val_losses['r0']:.6f} | W1: {val_losses['w1']:.6f}")


print("\n--- Training Complete ---")

# ===================================================================
# PHASE 3: FINAL EVALUATION (Full Multi-Head)
# ===================================================================

print("\n--- Phase 3: Final Evaluation ---")
model.eval()

# Store all predictions
all_preds = {'soh': [], 'l1': [], 'r0': [], 'r1': [], 'cpe1_0': [], 'cpe1_1': [], 'w1': []}
all_true = {'soh': [], 'params': []}

with torch.no_grad():
    # Use val_loader to get scaled validation data
    for (features_spec, features_scalar), true_params, true_soh in val_loader:
        features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
        
        pred_soh, pred_l1, pred_r0, pred_r1, pred_cpe1_0, pred_cpe1_1, pred_w1 = model(features_spec, features_scalar)
        
        # Add scaled predictions to lists
        all_preds['soh'].append(pred_soh.cpu())
        all_preds['l1'].append(pred_l1.cpu())
        all_preds['r0'].append(pred_r0.cpu())
        all_preds['r1'].append(pred_r1.cpu())
        all_preds['cpe1_0'].append(pred_cpe1_0.cpu()) # Added
        all_preds['cpe1_1'].append(pred_cpe1_1.cpu())
        all_preds['w1'].append(pred_w1.cpu())
        
        # Add scaled true values to lists
        all_true['soh'].append(true_soh.cpu())
        all_true['params'].append(true_params.cpu()) # true_params is already scaled

# Concatenate all batches
pred_soh_final = torch.cat(all_preds['soh']).numpy()
true_soh_final = torch.cat(all_true['soh']).numpy()
true_params_final_scaled = torch.cat(all_true['params']).numpy() # (N, 6)

# --- REWRITTEN: Cleaner De-scaling for 6 params ---
# Stack all scaled parameter predictions into (N, 6) array
pred_l1_scaled = torch.cat(all_preds['l1'])
pred_r0_scaled = torch.cat(all_preds['r0'])
pred_r1_scaled = torch.cat(all_preds['r1'])
pred_cpe1_0_scaled = torch.cat(all_preds['cpe1_0']) # Added
pred_cpe1_1_scaled = torch.cat(all_preds['cpe1_1'])
pred_w1_scaled = torch.cat(all_preds['w1'])

pred_params_final_scaled = torch.cat(
    [pred_l1_scaled, pred_r0_scaled, pred_r1_scaled, pred_cpe1_0_scaled, pred_cpe1_1_scaled, pred_w1_scaled], 
    dim=1
).numpy() # (N, 6)

# De-scale the true and predicted parameter arrays in one shot
true_params_final_descaled = scaler_y_params.inverse_transform(true_params_final_scaled)
pred_params_final_descaled = scaler_y_params.inverse_transform(pred_params_final_scaled)

# --- Calculate Final Errors ---
print(f"\n--- Final Model Evaluation ---")
soh_rmse = np.sqrt(mean_squared_error(true_soh_final, pred_soh_final))
print(f"SoH Prediction RMSE: {soh_rmse*100:.2f}% SoH")

# --- ADDED: Create the RMSE dictionary ---
final_rmse_dict = {'SoH': soh_rmse}
# ---

# param_columns is now the full 6-item list
for i, name in enumerate(param_columns):
    true_val = true_params_final_descaled[:, i]
    pred_val = pred_params_final_descaled[:, i]
    param_rmse = np.sqrt(mean_squared_error(true_val, pred_val))
    print(f"  - {name} RMSE: {param_rmse:.4f}")
    
    # --- ADDED: Add param RMSE to dict ---
    final_rmse_dict[name] = param_rmse
    # ---

# --- ADDED: Define a unique name and save the results ---
# !! CHANGE THIS NAME FOR EACH EXPERIMENT !!
exp_name = "CNN_7_Tower_v2_LR0.0005_Epochs500" 
model_arch_name = "CNN_7_Tower"

experiment_tracker.save_experiment_results(
    experiment_name=exp_name,
    model_architecture=model_arch_name,
    final_rmse=final_rmse_dict,
    learning_curves=history
)
# --- End of additions ---

--- Phase 0: Preparing Data & Engineering Features ---


Processing Spectra: 100%|██████████| 549/549 [00:00<00:00, 2315.74it/s]


Data ready: X_spectrum ((549, 100)), X_scalar ((549, 8))
Targets ready: y_params ((549, 6)), y_soh ((549, 1))

--- Phase 1: Setting up PyTorch components ---
Train/Test split: X_scalar_train ((439, 8)), y_params_train ((439, 6))

--- Phase 2: Starting Model Training ---
Using device: cpu
Number of Frequency Points: 50
Model Input Sizes: 100 (spectrum) + 8 (scalar)
Using individual loss weights: {'soh': 1000.0, 'l1': 0.5, 'r0': 0.5, 'r1': 1.0, 'cpe1_0': 1.0, 'cpe1_1': 1.0, 'w1': 0.5}
Epoch [001/100] - Train Loss: 84.0613 | Val Loss: 16.6230
    Val SoH: 0.012944 | R1: 0.255952 | CPE1_0: 0.691696 | CPE1_1: 0.828073
    L1: 1.633604 | R0: 1.649580 | W1: 0.523884
Epoch [025/100] - Train Loss: 2.5133 | Val Loss: 3.5261
    Val SoH: 0.000263 | R1: 0.299260 | CPE1_0: 0.550223 | CPE1_1: 0.836718
    L1: 1.499221 | R0: 1.557737 | W1: 0.097237
Epoch [050/100] - Train Loss: 2.3515 | Val Loss: 3.4421
    Val SoH: 0.000200 | R1: 0.332187 | CPE1_0: 0.558261 | CPE1_1: 0.835515
    L1: 1.368643 | R0: 

## Transformer Model

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
import sys  # --- ADDED ---

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# --- ADDED: Import the tracker ---
tracker_path = r'/Users/sushankmishra/Desktop/MTP_Materials/EIS Fitting'
if tracker_path not in sys.path:
    sys.path.append(tracker_path)
import experiment_tracker
# ---

# ===================================================================
# --- COMMON PARAMETERS ---
# ===================================================================
all_battery_data_EIS = pd.read_csv('all_battery_data_with_EIS_params.csv')
# This is the 5-parameter list from your script
param_columns = ['L1', 'R0', 'R1', 'CPE1_1', 'W1']
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']

# ===================================================================
# PHASE 0: DATA & FEATURE ENGINEERING
# ===================================================================
# (This phase is identical for both models)

print("--- Phase 0: Preparing Data & Engineering Features ---")

# 1. Clean and align all data
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH', 'SoC']
).reset_index(drop=True)

# 2. Define fixed frequencies for features
fixed_freqs = np.logspace(-2, 4, 50) # 50 points

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    feat_SoC = row['SoC'] 
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)') 
        f_x, R, X = raw_df['Frequency(Hz)'].values, raw_df['R(ohm)'].values, raw_df['X(ohm)'].values
        Z_imag_neg = -X
        
        feat_R_ohmic = R[-1] 
        feat_R_low = R[0]
        indices, _ = find_peaks(Z_imag_neg, prominence=1e-4) 
        if len(indices) == 2:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = Z_imag_neg[indices[1]], f_x[indices[1]]
        elif len(indices) == 1:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0
        else:
            feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0, 0.0, 0.0
        plot_features_array = np.array([feat_R_ohmic, feat_R_low, feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq])
        
        Z_phase_deg = np.angle(R + 1j * X, deg=True)
        feat_phase_min, feat_phase_min_freq = np.min(Z_phase_deg), f_x[np.argmin(Z_phase_deg)]
        phase_features_array = np.array([feat_phase_min, feat_phase_min_freq])

        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        R_features, X_features = interp_R(fixed_freqs), interp_X(fixed_freqs)
        
        # This is your 9-feature scalar set
        features = np.concatenate([R_features, X_features, plot_features_array, phase_features_array, np.array([feat_SoC])])
        eis_features_list.append(features)
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None)

# 4. Create final aligned DataFrames
r_cols = [f'R_{i}' for i in range(len(fixed_freqs))]
x_cols = [f'X_{i}' for i in range(len(fixed_freqs))]
plot_feature_names = ['R_ohmic', 'R_low', 'Peak_LF_Height', 'Peak_LF_Freq', 'Peak_HF_Height', 'Peak_HF_Freq']
phase_feature_names = ['Phase_Min', 'Phase_Min_Freq']
context_feature_names = ['SoC']
all_feature_names = r_cols + x_cols + plot_feature_names + phase_feature_names + context_feature_names

X_features_df = pd.DataFrame(eis_features_list, columns=all_feature_names)
y_params_df = df_cleaned[param_columns]
y_soh_df = df_cleaned[['SoH']]

# 5. Drop failed rows
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

# --- 6. NEW: Split X into spectrum and scalar features ---
spec_cols = r_cols + x_cols
X_spec_final = X_features_final[spec_cols]
scalar_cols = plot_feature_names + phase_feature_names + context_feature_names # 9 features
X_scalar_final = X_features_final[scalar_cols]

print(f"Data ready: X_spectrum ({X_spec_final.shape}), X_scalar ({X_scalar_final.shape})")

# ===================================================================
# PHASE 1: PYTORCH SETUP (Model 2: Transformer)
# ===================================================================

print("\n--- Phase 1: Setting up PyTorch components (Transformer Model) ---")

# 1. Scale features
scaler_X_spec = StandardScaler()
scaler_X_scalar = StandardScaler()
scaler_y_params = StandardScaler()

X_spec_scaled = scaler_X_spec.fit_transform(X_spec_final)
X_scalar_scaled = scaler_X_scalar.fit_transform(X_scalar_final)
y_params_scaled = scaler_y_params.fit_transform(y_params_final) # (N, 5)
y_soh_values = y_soh_final.values

# 2. Train/Test Split
(X_spec_train, X_spec_val,
 X_scalar_train, X_scalar_val,
 y_params_train, y_params_val, 
 y_soh_train, y_soh_val) = train_test_split(
    X_spec_scaled, X_scalar_scaled, y_params_scaled, y_soh_values, 
    test_size=0.2, random_state=42
)

# 3. Custom PyTorch Dataset
class EISDataset(Dataset):
    def __init__(self, spec_features, scalar_features, params, soh):
        self.X_spec = torch.tensor(spec_features, dtype=torch.float32)
        self.X_scalar = torch.tensor(scalar_features, dtype=torch.float32)
        self.y_params = torch.tensor(params, dtype=torch.float32)
        self.y_soh = torch.tensor(soh, dtype=torch.float32)
    def __len__(self):
        return len(self.X_spec)
    def __getitem__(self, idx):
        return (self.X_spec[idx], self.X_scalar[idx]), self.y_params[idx], self.y_soh[idx]

train_dataset = EISDataset(X_spec_train, X_scalar_train, y_params_train, y_soh_train)
val_dataset = EISDataset(X_spec_val, X_scalar_val, y_params_val, y_soh_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 4. Define the SOTA Multi-Head Hybrid Transformer Model
class MultiHeadEISModel(nn.Module):
    def __init__(self, num_scalar_features, num_params,
                 d_model=128, nhead=8, num_encoder_layers=4, dim_feedforward=512):
        super(MultiHeadEISModel, self).__init__()
        
        # --- Branch 1: Transformer for Spectrum (100 features) ---
        self.patch_embed = nn.Conv1d(
            in_channels=2, 
            out_channels=d_model, 
            kernel_size=10, 
            stride=5
        ) # Shape -> (batch, 128, 9) [num_patches = (50-10)/5 + 1 = 9]
        
        num_patches = 9
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_encoder_layers
        )
        
        # --- Branch 2: MLP for Scalar Features (9 features) ---
        self.scalar_branch = nn.Sequential(
            nn.Linear(num_scalar_features, 64),
            nn.ReLU(),
            nn.Linear(64, d_model)
        )

        # --- Shared Body ---
        self.shared_body = nn.Sequential(
            nn.Linear(d_model + d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # --- Heads ---
        self.param_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_params) # num_params will be 5
        )
        self.soh_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x_spec, x_scalar):
        x_spec = x_spec.reshape(-1, 2, 50) 
        patches = self.patch_embed(x_spec) 
        patches = patches.permute(0, 2, 1) 
        B = x_spec.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1) 
        x = torch.cat((cls_tokens, patches), dim=1) 
        x = x + self.pos_embed
        transformer_out = self.transformer_encoder(x) 
        out_spec = transformer_out[:, 0, :] 
        out_scalar = self.scalar_branch(x_scalar) 
        merged = torch.cat((out_spec, out_scalar), dim=1) 
        shared_output = self.shared_body(merged)
        params_pred = self.param_head(shared_output)
        soh_pred = self.soh_head(shared_output)
        
        return params_pred, soh_pred

# ===================================================================
# PHASE 2: MODEL TRAINING
# ===================================================================

print("\n--- Phase 2: Starting Model Training ---")

# 1. Setup Model, Loss, Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

NUM_SCALAR_FEATURES = X_scalar_final.shape[1] # 9
NUM_PARAMS = y_params_final.shape[1] # 5
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

print(f"Model Input Sizes: {X_spec_final.shape[1]} (spectrum) + {NUM_SCALAR_FEATURES} (scalar)")

model = MultiHeadEISModel(NUM_SCALAR_FEATURES, NUM_PARAMS).to(device)
loss_params_fn = nn.MSELoss()
loss_soh_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Define your loss weights
alpha = 0.7 
beta = 0.3  

# --- ADDED: Initialize history ---
history = {
    'train_loss': [],
    'val_total': [],
    'val_soh': [],
    'val_params': []
}
# ---

# 3. Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    for (features_spec, features_scalar), true_params, true_soh in train_loader:
        features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
        true_params, true_soh = true_params.to(device), true_soh.to(device)
        
        pred_params, pred_soh = model(features_spec, features_scalar)
        
        loss_params = loss_params_fn(pred_params, true_params)
        loss_soh = loss_soh_fn(pred_soh, true_soh.view(-1, 1)) # Ensure true_soh is correct shape
        loss_total = (alpha * loss_soh) + (beta * loss_params)
        
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()
        total_train_loss += loss_total.item()

    # --- Validation ---
    model.eval()
    total_val_loss, total_val_soh_loss, total_val_params_loss = 0, 0, 0
    with torch.no_grad():
        for (features_spec, features_scalar), true_params, true_soh in val_loader:
            features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
            true_params, true_soh = true_params.to(device), true_soh.to(device)
            
            pred_params, pred_soh = model(features_spec, features_scalar)
            
            loss_params = loss_params_fn(pred_params, true_params)
            loss_soh = loss_soh_fn(pred_soh, true_soh.view(-1, 1)) # Ensure true_soh is correct shape
            loss_total = (alpha * loss_soh) + (beta * loss_params)
            
            total_val_loss += loss_total.item()
            total_val_soh_loss += loss_soh.item()
            total_val_params_loss += loss_params.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    avg_soh_loss = total_val_soh_loss / len(val_loader)
    avg_params_loss = total_val_params_loss / len(val_loader)
    
    # --- ADDED: Save to history dict ---
    history['train_loss'].append(avg_train_loss)
    history['val_total'].append(avg_val_loss)
    history['val_soh'].append(avg_soh_loss)
    history['val_params'].append(avg_params_loss)
    # ---

    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} (SoH: {avg_soh_loss:.6f}, Params: {avg_params_loss:.6f})")

print("\n--- Training Complete ---")

# ===================================================================
# PHASE 3: FINAL EVALUATION
# ===================================================================
model.eval()
with torch.no_grad():
    # Use the validation set data (already scaled)
    all_spec_features = torch.tensor(X_spec_val, dtype=torch.float32).to(device)
    all_scalar_features = torch.tensor(X_scalar_val, dtype=torch.float32).to(device)
    
    pred_params_scaled, pred_soh = model(all_spec_features, all_scalar_features)
    
    # De-scale predictions
    pred_params = scaler_y_params.inverse_transform(pred_params_scaled.cpu().numpy())
    # De-scale true validation values
    true_params_descaled = scaler_y_params.inverse_transform(y_params_val)
    
    # Calculate SoH RMSE (no de-scaling needed for y_soh_val)
    soh_rmse = np.sqrt(mean_squared_error(y_soh_val, pred_soh.cpu().numpy()))
    
    print(f"\n--- Final Model Evaluation ---")
    print(f"SoH Prediction RMSE: {soh_rmse*100:.2f}% SoH")
    
    # --- ADDED: Initialize RMSE dict ---
    final_rmse_dict = {'SoH': soh_rmse}
    # ---

    for i, name in enumerate(param_columns):
        param_rmse = np.sqrt(mean_squared_error(true_params_descaled[:, i], pred_params[:, i]))
        print(f"  - {name} RMSE: {param_rmse:.4f}")
        
        # --- ADDED: Add param to dict ---
        final_rmse_dict[name] = param_rmse
        # ---

# --- ADDED: Define a unique name and save the results ---
# !! CHANGE THIS NAME FOR EACH EXPERIMENT !!
exp_name = "Transformer_v1_LR0.001_Epochs50" 
model_arch_name = "Transformer_SharedBody_9Features"

try:
    print("\n--- Saving learning curves to JSON file ---")
    experiment_tracker.save_experiment_results(
        experiment_name=exp_name,
        model_architecture=model_arch_name,
        final_rmse=final_rmse_dict,
        learning_curves=history
    )
    print("--- JSON save complete. ---")
except Exception as e:
    print(f"--- !! WARNING: JSON tracker save failed: {e} !! ---")
# --- End of additions ---

--- Phase 0: Preparing Data & Engineering Features ---


Processing Spectra: 100%|██████████| 549/549 [00:00<00:00, 2468.37it/s]


Data ready: X_spectrum ((549, 100)), X_scalar ((549, 9))

--- Phase 1: Setting up PyTorch components (Transformer Model) ---

--- Phase 2: Starting Model Training ---
Using device: cpu
Model Input Sizes: 100 (spectrum) + 9 (scalar)
Epoch [01/100] - Train Loss: 0.302456 | Val Loss: 0.297180 (SoH: 0.006512, Params: 0.975404)
Epoch [02/100] - Train Loss: 0.244412 | Val Loss: 0.271202 (SoH: 0.001503, Params: 0.900500)
Epoch [03/100] - Train Loss: 0.207644 | Val Loss: 0.272535 (SoH: 0.001906, Params: 0.904005)
Epoch [04/100] - Train Loss: 0.204346 | Val Loss: 0.269770 (SoH: 0.001995, Params: 0.894579)
Epoch [05/100] - Train Loss: 0.195351 | Val Loss: 0.268094 (SoH: 0.001106, Params: 0.891067)
Epoch [06/100] - Train Loss: 0.195975 | Val Loss: 0.270967 (SoH: 0.001711, Params: 0.899230)
Epoch [07/100] - Train Loss: 0.194245 | Val Loss: 0.266750 (SoH: 0.001952, Params: 0.884612)
Epoch [08/100] - Train Loss: 0.192024 | Val Loss: 0.264690 (SoH: 0.001399, Params: 0.879035)
Epoch [09/100] - Train L

## ResNet Model

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ===================================================================
# --- COMMON PARAMETERS ---
# ===================================================================
all_battery_data_EIS = pd.read_csv('all_battery_data_with_EIS_params.csv')
param_columns = ['L1', 'R0', 'R1', 'CPE1_1', 'W1']
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']

# ===================================================================
# PHASE 0: DATA & FEATURE ENGINEERING
# ===================================================================

print("--- Phase 0: Preparing Data & Engineering Features ---")

# 1. Clean and align all data
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH', 'SoC']
).reset_index(drop=True)

# 2. Define fixed frequencies for features
fixed_freqs = np.logspace(-2, 4, 50) # 50 points

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    feat_SoC = row['SoC'] 
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)') 
        f_x, R, X = raw_df['Frequency(Hz)'].values, raw_df['R(ohm)'].values, raw_df['X(ohm)'].values
        Z_imag_neg = -X
        
        feat_R_ohmic = R[-1] 
        feat_R_low = R[0]
        indices, _ = find_peaks(Z_imag_neg, prominence=1e-4) 
        if len(indices) == 2:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = Z_imag_neg[indices[1]], f_x[indices[1]]
        elif len(indices) == 1:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0
        else:
            feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0, 0.0, 0.0
        plot_features_array = np.array([feat_R_ohmic, feat_R_low, feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq])
        
        Z_phase_deg = np.angle(R + 1j * X, deg=True)
        feat_phase_min, feat_phase_min_freq = np.min(Z_phase_deg), f_x[np.argmin(Z_phase_deg)]
        phase_features_array = np.array([feat_phase_min, feat_phase_min_freq])

        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        R_features, X_features = interp_R(fixed_freqs), interp_X(fixed_freqs)
        
        features = np.concatenate([R_features, X_features, plot_features_array, phase_features_array, np.array([feat_SoC])])
        eis_features_list.append(features)
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None)

# 4. Create final aligned DataFrames
r_cols = [f'R_{i}' for i in range(len(fixed_freqs))]
x_cols = [f'X_{i}' for i in range(len(fixed_freqs))]
plot_feature_names = ['R_ohmic', 'R_low', 'Peak_LF_Height', 'Peak_LF_Freq', 'Peak_HF_Height', 'Peak_HF_Freq']
phase_feature_names = ['Phase_Min', 'Phase_Min_Freq']
context_feature_names = ['SoC']
all_feature_names = r_cols + x_cols + plot_feature_names + phase_feature_names + context_feature_names

X_features_df = pd.DataFrame(eis_features_list, columns=all_feature_names)
y_params_df = df_cleaned[param_columns]
y_soh_df = df_cleaned[['SoH']]

# 5. Drop failed rows
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

# --- 6. Split X into spectrum (100) and scalar (9) features ---
spec_cols = r_cols + x_cols
X_spec_final = X_features_final[spec_cols]
scalar_cols = plot_feature_names + phase_feature_names + context_feature_names
X_scalar_final = X_features_final[scalar_cols]

# ===================================================================
# PHASE 1: PYTORCH SETUP (Dataset & SOTA Model)
# ===================================================================

print("\n--- Phase 1: Setting up PyTorch components ---")

# 3. Scale each DataFrame separately
scaler_X_spec = StandardScaler()
scaler_X_scalar = StandardScaler()
scaler_y_params = StandardScaler()

X_spec_scaled = scaler_X_spec.fit_transform(X_spec_final)
X_scalar_scaled = scaler_X_scalar.fit_transform(X_scalar_final)
y_params_scaled = scaler_y_params.fit_transform(y_params_final)
y_soh_values = y_soh_final.values 

# 4. Train/Test Split (using the two new X variables)
(X_spec_train, X_spec_val,
 X_scalar_train, X_scalar_val,
 y_params_train, y_params_val, 
 y_soh_train, y_soh_val) = train_test_split(
    X_spec_scaled, X_scalar_scaled, y_params_scaled, y_soh_values, 
    test_size=0.2, random_state=42
)
# 4. Custom PyTorch Dataset (This is correct)
class EISDataset(Dataset):
    def __init__(self, spec_features, scalar_features, params, soh):
        # spec_features will be (N, 100)
        self.X_spec = torch.tensor(spec_features, dtype=torch.float32) 
        # scalar_features will be (N, 9)
        self.X_scalar = torch.tensor(scalar_features, dtype=torch.float32) 
        self.y_params = torch.tensor(params, dtype=torch.float32)
        self.y_soh = torch.tensor(soh, dtype=torch.float32)

    def __len__(self):
        return len(self.X_spec)

    def __getitem__(self, idx):
        # This will now correctly return one (100,) tensor and one (9,) tensor
        return (self.X_spec[idx], self.X_scalar[idx]), self.y_params[idx], self.y_soh[idx]

train_dataset = EISDataset(X_spec_train, X_scalar_train, y_params_train, y_soh_train)
val_dataset = EISDataset(X_spec_val, X_scalar_val, y_params_val, y_soh_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 6. Define the SOTA Multi-Head Hybrid CNN-MLP Model
# (This model is correct and does not need to change)
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = self.relu(out)
        return out

class DeepResidualEISModel(nn.Module):
    def __init__(self, num_scalar_features, num_params, num_freq_points):
        super(DeepResidualEISModel, self).__init__()
        
        self.num_freq_points = num_freq_points
        
        # --- Enhanced CNN Branch with Residual Blocks ---
        self.cnn_branch = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            ResidualBlock(32),
            nn.MaxPool1d(kernel_size=2),
            
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            ResidualBlock(64),
            nn.MaxPool1d(kernel_size=2),
            
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(8),  # Adaptive pooling to fixed size
            nn.Flatten()
        )
        
        # --- Enhanced Scalar Branch ---
        self.scalar_branch = nn.Sequential(
            nn.Linear(num_scalar_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        
        merged_size = 128 * 8 + 128  # CNN output + scalar output
        
        # --- Separate towers with more capacity ---
        self.param_tower = nn.Sequential(
            nn.Linear(merged_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_params)
        )
        
        self.soh_tower = nn.Sequential(
            nn.Linear(merged_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x_spec, x_scalar):
        x_spec = x_spec.reshape(-1, 2, self.num_freq_points)
        
        out_cnn = self.cnn_branch(x_spec)
        out_scalar = self.scalar_branch(x_scalar)
        
        merged = torch.cat((out_cnn, out_scalar), dim=1)
        
        params_pred = self.param_tower(merged)
        soh_pred = self.soh_tower(merged)
        
        return params_pred, soh_pred
# ===================================================================
# PHASE 2: MODEL TRAINING
# ===================================================================

print("\n--- Phase 2: Starting Model Training ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Dynamically get input sizes
NUM_FREQ_POINTS = len(fixed_freqs) 
NUM_SCALAR_FEATURES = X_scalar_final.shape[1] 
NUM_PARAMS = y_params_final.shape[1]
NUM_EPOCHS = 300
LEARNING_RATE = 0.0001
print(f"Number of Frequency Points: {NUM_FREQ_POINTS}")
print(f"Model Input Sizes: {NUM_FREQ_POINTS*2} (spectrum) + {NUM_SCALAR_FEATURES} (scalar)")

model = DeepResidualEISModel(NUM_SCALAR_FEATURES, NUM_PARAMS, NUM_FREQ_POINTS).to(device)
loss_params_fn = nn.MSELoss()
loss_soh_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

alpha = 1000.0  # Make the SoH loss "louder"
beta = 1.0

# Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    
    for (features_spec, features_scalar), true_params, true_soh in train_loader:
        features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
        true_params, true_soh = true_params.to(device), true_soh.to(device)
        pred_params, pred_soh = model(features_spec, features_scalar)
        
        loss_params = loss_params_fn(pred_params, true_params)
        loss_soh = loss_soh_fn(pred_soh, true_soh)
        loss_total = (alpha * loss_soh) + (beta * loss_params)
        
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()
        total_train_loss += loss_total.item()

    # Validation Loop
    model.eval()
    total_val_loss, total_val_soh_loss, total_val_params_loss = 0, 0, 0
    with torch.no_grad():
        for (features_spec, features_scalar), true_params, true_soh in val_loader:
            features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
            true_params, true_soh = true_params.to(device), true_soh.to(device)
            
            pred_params, pred_soh = model(features_spec, features_scalar)
            
            loss_params = loss_params_fn(pred_params, true_params)
            loss_soh = loss_soh_fn(pred_soh, true_soh)
            loss_total = (alpha * loss_soh) + (beta * loss_params)
            
            total_val_loss += loss_total.item()
            total_val_soh_loss += loss_soh.item()
            total_val_params_loss += loss_params.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    avg_soh_loss = total_val_soh_loss / len(val_loader)
    avg_params_loss = total_val_params_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} (SoH: {avg_soh_loss:.6f}, Params: {avg_params_loss:.6f})")

print("\n--- Training Complete ---")

# ===================================================================
# PHASE 3: FINAL EVALUATION
# ===================================================================
model.eval()
with torch.no_grad():
    all_spec_features = torch.tensor(X_spec_val, dtype=torch.float32).to(device)
    all_scalar_features = torch.tensor(X_scalar_val, dtype=torch.float32).to(device)
    
    pred_params_scaled, pred_soh = model(all_spec_features, all_scalar_features)
    
    # De-scale the predictions
    pred_params = scaler_y_params.inverse_transform(pred_params_scaled.cpu().numpy())
    true_params_descaled = scaler_y_params.inverse_transform(y_params_val)
    
    # Calculate Final Errors
    soh_rmse = np.sqrt(mean_squared_error(y_soh_val, pred_soh.cpu().numpy()))
    
    print(f"\n--- Final Model Evaluation ---")
    print(f"SoH Prediction RMSE: {soh_rmse*100:.2f}% SoH")
    
    for i, name in enumerate(param_columns):
        param_rmse = np.sqrt(mean_squared_error(true_params_descaled[:, i], pred_params[:, i]))
        print(f"  - {name} RMSE: {param_rmse:.4f}")

--- Phase 0: Preparing Data & Engineering Features ---


Processing Spectra: 100%|██████████| 549/549 [00:00<00:00, 1570.71it/s]



--- Phase 1: Setting up PyTorch components ---

--- Phase 2: Starting Model Training ---
Using device: cpu
Number of Frequency Points: 50
Model Input Sizes: 100 (spectrum) + 9 (scalar)
Epoch [01/300] - Train Loss: 146.897472 | Val Loss: 159.659756 (SoH: 0.158474, Params: 1.185282)
Epoch [02/300] - Train Loss: 123.192338 | Val Loss: 123.581663 (SoH: 0.122475, Params: 1.107021)
Epoch [03/300] - Train Loss: 107.125142 | Val Loss: 114.563984 (SoH: 0.113577, Params: 0.986789)
Epoch [04/300] - Train Loss: 92.517026 | Val Loss: 105.092297 (SoH: 0.104156, Params: 0.936002)
Epoch [05/300] - Train Loss: 79.529532 | Val Loss: 90.317497 (SoH: 0.089426, Params: 0.891067)
Epoch [06/300] - Train Loss: 69.573096 | Val Loss: 82.801233 (SoH: 0.081911, Params: 0.889990)
Epoch [07/300] - Train Loss: 61.504565 | Val Loss: 79.197966 (SoH: 0.078310, Params: 0.887560)
Epoch [08/300] - Train Loss: 55.959302 | Val Loss: 64.547673 (SoH: 0.063675, Params: 0.872996)
Epoch [09/300] - Train Loss: 46.545873 | Val Lo

## Cross Stitch Networks


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ===================================================================
# --- COMMON PARAMETERS ---
# ===================================================================
all_battery_data_EIS = pd.read_csv('all_battery_data_with_EIS_params.csv')
param_columns = ['L1', 'R0', 'R1', 'CPE1_0', 'CPE1_1', 'W1']
col_names = ['Frequency(Hz)', 'R(ohm)', 'X(ohm)']

# ===================================================================
# PHASE 0: DATA & FEATURE ENGINEERING
# ===================================================================

print("--- Phase 0: Preparing Data & Engineering Features ---")

# 1. Clean and align all data
df_cleaned = all_battery_data_EIS.dropna(
    subset=param_columns + ['SoH', 'SoC']
).reset_index(drop=True)

# 2. Define fixed frequencies for features
fixed_freqs = np.logspace(-2, 4, 50) # 50 points

# 3. Loop through files and engineer X features
eis_features_list = []
for index, row in tqdm(df_cleaned.iterrows(), total=len(df_cleaned), desc="Processing Spectra"):
    f = row['EIS_File']
    feat_SoC = row['SoC'] 
    try:
        raw_df = pd.read_csv(f, sep='\t', names=col_names)
        raw_df = raw_df.sort_values(by='Frequency(Hz)') 
        f_x, R, X = raw_df['Frequency(Hz)'].values, raw_df['R(ohm)'].values, raw_df['X(ohm)'].values
        Z_imag_neg = -X
        
        feat_R_ohmic = R[-1] 
        feat_R_low = R[0]
        indices, _ = find_peaks(Z_imag_neg, prominence=1e-4) 
        if len(indices) == 2:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = Z_imag_neg[indices[1]], f_x[indices[1]]
        elif len(indices) == 1:
            feat_Peak_LF_Height, feat_Peak_LF_Freq = Z_imag_neg[indices[0]], f_x[indices[0]]
            feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0
        else:
            feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq = 0.0, 0.0, 0.0, 0.0
        plot_features_array = np.array([feat_R_ohmic, feat_R_low, feat_Peak_LF_Height, feat_Peak_LF_Freq, feat_Peak_HF_Height, feat_Peak_HF_Freq])
        
        Z_phase_deg = np.angle(R + 1j * X, deg=True)
        feat_phase_min, feat_phase_min_freq = np.min(Z_phase_deg), f_x[np.argmin(Z_phase_deg)]
        phase_features_array = np.array([feat_phase_min, feat_phase_min_freq])

        interp_R = interp1d(f_x, R, bounds_error=False, fill_value='extrapolate')
        interp_X = interp1d(f_x, X, bounds_error=False, fill_value='extrapolate')
        R_features, X_features = interp_R(fixed_freqs), interp_X(fixed_freqs)
        
        features = np.concatenate([R_features, X_features, plot_features_array, phase_features_array, np.array([feat_SoC])])
        eis_features_list.append(features)
    except Exception as e:
        tqdm.write(f"Warning: Could not process file {f}. Error: {e}")
        eis_features_list.append(None)

# 4. Create final aligned DataFrames
r_cols = [f'R_{i}' for i in range(len(fixed_freqs))]
x_cols = [f'X_{i}' for i in range(len(fixed_freqs))]
plot_feature_names = ['R_ohmic', 'R_low', 'Peak_LF_Height', 'Peak_LF_Freq', 'Peak_HF_Height', 'Peak_HF_Freq']
phase_feature_names = ['Phase_Min', 'Phase_Min_Freq']
context_feature_names = ['SoC']
all_feature_names = r_cols + x_cols + plot_feature_names + phase_feature_names + context_feature_names

X_features_df = pd.DataFrame(eis_features_list, columns=all_feature_names)
y_params_df = df_cleaned[param_columns]
y_soh_df = df_cleaned[['SoH']]

# 5. Drop failed rows
failed_indices = X_features_df[X_features_df.isnull().any(axis=1)].index
X_features_final = X_features_df.drop(failed_indices)
y_params_final = y_params_df.drop(failed_indices)
y_soh_final = y_soh_df.drop(failed_indices)

# --- 6. Split X into spectrum (100) and scalar (9) features ---
spec_cols = r_cols + x_cols
X_spec_final = X_features_final[spec_cols]
scalar_cols = plot_feature_names + phase_feature_names + context_feature_names
X_scalar_final = X_features_final[scalar_cols]

# ===================================================================
# PHASE 1: PYTORCH SETUP (Dataset & SOTA Model)
# ===================================================================

print("\n--- Phase 1: Setting up PyTorch components ---")

# 3. Scale each DataFrame separately
scaler_X_spec = StandardScaler()
scaler_X_scalar = StandardScaler()
scaler_y_params = StandardScaler()

X_spec_scaled = scaler_X_spec.fit_transform(X_spec_final)
X_scalar_scaled = scaler_X_scalar.fit_transform(X_scalar_final)
y_params_scaled = scaler_y_params.fit_transform(y_params_final)
y_soh_values = y_soh_final.values 

# 4. Train/Test Split (using the two new X variables)
(X_spec_train, X_spec_val,
 X_scalar_train, X_scalar_val,
 y_params_train, y_params_val, 
 y_soh_train, y_soh_val) = train_test_split(
    X_spec_scaled, X_scalar_scaled, y_params_scaled, y_soh_values, 
    test_size=0.2, random_state=42
)
# 4. Custom PyTorch Dataset (This is correct)
class EISDataset(Dataset):
    def __init__(self, spec_features, scalar_features, params, soh):
        # spec_features will be (N, 100)
        self.X_spec = torch.tensor(spec_features, dtype=torch.float32) 
        # scalar_features will be (N, 9)
        self.X_scalar = torch.tensor(scalar_features, dtype=torch.float32) 
        self.y_params = torch.tensor(params, dtype=torch.float32)
        self.y_soh = torch.tensor(soh, dtype=torch.float32)

    def __len__(self):
        return len(self.X_spec)

    def __getitem__(self, idx):
        # This will now correctly return one (100,) tensor and one (9,) tensor
        return (self.X_spec[idx], self.X_scalar[idx]), self.y_params[idx], self.y_soh[idx]

train_dataset = EISDataset(X_spec_train, X_scalar_train, y_params_train, y_soh_train)
val_dataset = EISDataset(X_spec_val, X_scalar_val, y_params_val, y_soh_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 6. Define the SOTA Multi-Head Hybrid CNN-MLP Model
# (This model is correct and does not need to change)
class CrossStitchUnit(nn.Module):
    def __init__(self, num_tasks=2):
        super(CrossStitchUnit, self).__init__()
        # Learnable cross-stitch matrix
        self.cross_stitch = nn.Parameter(torch.eye(num_tasks))
    
    def forward(self, task_features):
        # task_features: list of tensors [task1_features, task2_features]
        stacked = torch.stack(task_features, dim=0)  # (num_tasks, batch, features)
        mixed = torch.einsum('ij,jbf->ibf', self.cross_stitch, stacked)
        return [mixed[i] for i in range(len(task_features))]

class CrossStitchEISModel(nn.Module):
    def __init__(self, num_scalar_features, num_params, num_freq_points):
        super(CrossStitchEISModel, self).__init__()
        
        self.num_freq_points = num_freq_points
        
        # --- Shared feature extraction ---
        self.cnn_branch = nn.Sequential(
            nn.Conv1d(2, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten()
        )
        
        self.scalar_branch = nn.Sequential(
            nn.Linear(num_scalar_features, 64),
            nn.ReLU()
        )
        
        cnn_out_size = 128 * (num_freq_points // 4)
        merged_size = cnn_out_size + 64
        
        # --- Shared layers with cross-stitch ---
        self.shared1_params = nn.Linear(merged_size, 256)
        self.shared1_soh = nn.Linear(merged_size, 256)
        self.cross_stitch1 = CrossStitchUnit(2)
        
        self.shared2_params = nn.Linear(256, 128)
        self.shared2_soh = nn.Linear(256, 128)
        self.cross_stitch2 = CrossStitchUnit(2)
        
        # --- Task-specific heads ---
        self.param_head = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_params)
        )
        
        self.soh_head = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x_spec, x_scalar):
        x_spec = x_spec.reshape(-1, 2, self.num_freq_points)
        
        out_cnn = self.cnn_branch(x_spec)
        out_scalar = self.scalar_branch(x_scalar)
        merged = torch.cat((out_cnn, out_scalar), dim=1)
        
        # First shared layer with cross-stitch
        h1_params = self.shared1_params(merged)
        h1_soh = self.shared1_soh(merged)
        h1_params, h1_soh = self.cross_stitch1([h1_params, h1_soh])
        
        # Second shared layer with cross-stitch
        h2_params = self.shared2_params(h1_params)
        h2_soh = self.shared2_soh(h1_soh)
        h2_params, h2_soh = self.cross_stitch2([h2_params, h2_soh])
        
        # Task-specific predictions
        params_pred = self.param_head(h2_params)
        soh_pred = self.soh_head(h2_soh)
        
        return params_pred, soh_pred

# ===================================================================
# PHASE 2: MODEL TRAINING
# ===================================================================

print("\n--- Phase 2: Starting Model Training ---")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Dynamically get input sizes
NUM_FREQ_POINTS = len(fixed_freqs) 
NUM_SCALAR_FEATURES = X_scalar_final.shape[1] 
NUM_PARAMS = y_params_final.shape[1]
NUM_EPOCHS = 1000
LEARNING_RATE = 0.0001
print(f"Number of Frequency Points: {NUM_FREQ_POINTS}")
print(f"Model Input Sizes: {NUM_FREQ_POINTS*2} (spectrum) + {NUM_SCALAR_FEATURES} (scalar)")

model = CrossStitchEISModel(NUM_SCALAR_FEATURES, NUM_PARAMS, NUM_FREQ_POINTS).to(device)
loss_params_fn = nn.MSELoss()
loss_soh_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

alpha = 1000.0  # Make the SoH loss "louder"
beta = 1.0

# Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0
    
    for (features_spec, features_scalar), true_params, true_soh in train_loader:
        features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
        true_params, true_soh = true_params.to(device), true_soh.to(device)
        pred_params, pred_soh = model(features_spec, features_scalar)
        
        loss_params = loss_params_fn(pred_params, true_params)
        loss_soh = loss_soh_fn(pred_soh, true_soh)
        loss_total = (alpha * loss_soh) + (beta * loss_params)
        
        optimizer.zero_grad()
        loss_total.backward()
        optimizer.step()
        total_train_loss += loss_total.item()

    # Validation Loop
    model.eval()
    total_val_loss, total_val_soh_loss, total_val_params_loss = 0, 0, 0
    with torch.no_grad():
        for (features_spec, features_scalar), true_params, true_soh in val_loader:
            features_spec, features_scalar = features_spec.to(device), features_scalar.to(device)
            true_params, true_soh = true_params.to(device), true_soh.to(device)
            
            pred_params, pred_soh = model(features_spec, features_scalar)
            
            loss_params = loss_params_fn(pred_params, true_params)
            loss_soh = loss_soh_fn(pred_soh, true_soh)
            loss_total = (alpha * loss_soh) + (beta * loss_params)
            
            total_val_loss += loss_total.item()
            total_val_soh_loss += loss_soh.item()
            total_val_params_loss += loss_params.item()
            
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    avg_soh_loss = total_val_soh_loss / len(val_loader)
    avg_params_loss = total_val_params_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} (SoH: {avg_soh_loss:.6f}, Params: {avg_params_loss:.6f})")

print("\n--- Training Complete ---")

# ===================================================================
# PHASE 3: FINAL EVALUATION
# ===================================================================
model.eval()
with torch.no_grad():
    all_spec_features = torch.tensor(X_spec_val, dtype=torch.float32).to(device)
    all_scalar_features = torch.tensor(X_scalar_val, dtype=torch.float32).to(device)
    
    pred_params_scaled, pred_soh = model(all_spec_features, all_scalar_features)
    
    # De-scale the predictions
    pred_params = scaler_y_params.inverse_transform(pred_params_scaled.cpu().numpy())
    true_params_descaled = scaler_y_params.inverse_transform(y_params_val)
    
    # Calculate Final Errors
    soh_rmse = np.sqrt(mean_squared_error(y_soh_val, pred_soh.cpu().numpy()))
    
    print(f"\n--- Final Model Evaluation ---")
    print(f"SoH Prediction RMSE: {soh_rmse*100:.2f}% SoH")
    
    for i, name in enumerate(param_columns):
        param_rmse = np.sqrt(mean_squared_error(true_params_descaled[:, i], pred_params[:, i]))
        print(f"  - {name} RMSE: {param_rmse:.4f}")

# Model Comparison